<a href="https://colab.research.google.com/github/E-HAZMATs/FT-SmolLM/blob/main/FT_SmolLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip -qqq install transformers datasets trl torch
! pip -qqq install peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.8 MB/s eta 0:00:00


In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

In [94]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [11]:
dataset_name = 'HuggingFaceTB/smoltalk2'
ds = load_dataset(dataset_name, 'SFT', streaming=True)

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

In [16]:
ds.keys()

dict_keys(['LongAlign_64k_Qwen3_32B_yarn_131k_think', 'OpenThoughts3_1.2M_think', 'aya_dataset_Qwen3_32B_think', 'multi_turn_reasoning_if_think', 's1k_1.1_think', 'smolagents_toolcalling_traces_think', 'smoltalk_everyday_convs_reasoning_Qwen3_32B_think', 'smoltalk_multilingual8_Qwen3_32B_think', 'smoltalk_systemchats_Qwen3_32B_think', 'table_gpt_Qwen3_32B_think', 'LongAlign_64k_context_lang_annotated_lang_6_no_think', 'Mixture_of_Thoughts_science_no_think', 'OpenHermes_2.5_no_think', 'OpenThoughts3_1.2M_no_think_no_think', 'hermes_function_calling_v1_no_think', 'smoltalk_multilingual_8languages_lang_5_no_think', 'smoltalk_smollm3_everyday_conversations_no_think', 'smoltalk_smollm3_explore_instruct_rewriting_no_think', 'smoltalk_smollm3_smol_magpie_ultra_no_think', 'smoltalk_smollm3_smol_rewrite_no_think', 'smoltalk_smollm3_smol_summarize_no_think', 'smoltalk_smollm3_systemchats_30k_no_think', 'table_gpt_no_think', 'tulu_3_sft_personas_instruction_following_no_think', 'xlam_traces_no_th

In [55]:
split = 'Mixture_of_Thoughts_science_no_think'

In [96]:
model_name = "HuggingFaceTB/SmolLM-135M"
base = AutoModelForCausalLM.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name + '-Instruct')

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [56]:
print(f"Model Size: {(base.get_memory_footprint() / 1024**3):.3f}GB")

Model Size: 0.251GB


In [86]:
from datasets import Dataset
examples_list = list(ds[split].take(5))
examples = Dataset.from_list(examples_list)

In [78]:
tokenizer.chat_template

"{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"

In [87]:
def format_ds(batch):
    return {'text': [tokenizer.apply_chat_template(ex, tokenize=False) for ex in batch['messages']]}

text = examples.map(format_ds, batched=True, remove_columns=examples.column_names)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [88]:
text[0]

{'text': "<|im_start|>user\nWhat hormone's action is inhibited by caffeine, leading to increased urination?A: ADH\nB: Insulin\nC: Thyroxine\nD: Cortisol<|im_end|>\n<|im_start|>assistant\nCaffeine acts as a diuretic by inhibiting the action of antidiuretic hormone (ADH), which is responsible for signaling the kidneys to reabsorb water and concentrate urine. When ADH is suppressed, the kidneys excrete more water, leading to increased urination. The other hormones listed—insulin (regulates blood sugar), thyroxine (regulates metabolism), and cortisol (involved in stress response)—are not directly related to fluid balance or diuresis. \n\n**Answer: A**  \n\\boxed{A}<|im_end|>\n"}

In [90]:
prompt = "I'd like a recipe for something warm."
tokenized = tokenizer(prompt, return_tensors='pt')

with torch.no_grad():
  outputs = base.generate(
      **tokenized,
      max_new_tokens=200,
      temprature=0.7,
      do_sample=True,
      pad_token=tokenizer.eos_token_id
  )
  decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
  print(decoded)



ValueError: The following `model_kwargs` are not used by the model: ['temprature', 'pad_token'] (note: typos in the generate arguments will also show up in this list)